# Mathys — Étape 1 : étude de la courbure et de la Hessienne en 2D

## Sommaire

1. [Contexte et outils](#contexte)
2. [Étude de $L = 0.1\,\theta_1^2 + 10\,\theta_2^2$](#etude-loss)
3. [Rôles du gradient et de la Hessienne](#gradient-hessienne)
4. [Mesurer la platitude avec $H$](#platitude)
   1. [La plus grande valeur propre $\lambda_{\max}$](#lambda-max)
   2. [La trace $\mathrm{tr}(H)$](#trace)
   3. [Le score $\tilde B$ de *Flat Minima*](#score-b)
   4. [Récapitulatif](#recap)
5. [Sensibilité de la loss selon la direction](#sensibilite)

<a id="contexte"></a>
## 1. Contexte et outils

- Une image de taille $N \times N$, 1 canal (niveaux de gris)
- Convolution de fenêtre $1 \times 1$ : $h_p = \theta_1\, x_p + \theta_2$
- Global Average Pooling (GAP) : $\hat y = $ moyenne des $N^2$ pixels de $h$
- Loss : $L = 10\,\mathbb{E}\big[(\hat y - y)^2\big]$, avec la cible $y = 0$

### Pipeline

$$
\underbrace{x}_{N \times N \times 1}
\;\xrightarrow{\text{conv } 1\times 1:\ h_p = \theta_1 x_p + \theta_2}\;
\underbrace{h}_{N \times N \times 1}
\;\xrightarrow{\text{GAP}}\;
\hat y = \frac{1}{N^2}\sum_{p} h_p
\;\longrightarrow\;
L = 10\,\mathbb{E}\big[(\hat y - y)^2\big]
$$

Les pixels sont centrés (normalisés) : $x_p$ peut donc être négatif.

### Étape clé

Soit $z = \dfrac{1}{N^2}\sum_p x_p$ la moyenne de l'image d'entrée. Alors

$$\hat y = \frac{1}{N^2}\sum_p (\theta_1 x_p + \theta_2) = \theta_1\, z + \theta_2 .$$

Chaque image donne son propre $z$. Le jeu de données est construit pour que $\mathbb{E}[z] = 0$ et $\mathrm{Var}(z) = 0.01$.

### Loss

$$
L = 10\,\mathbb{E}\big[(\theta_1 z + \theta_2)^2\big]
= 10\big(\theta_1^2\,\mathbb{E}[z^2] + 2\theta_1\theta_2\,\mathbb{E}[z] + \theta_2^2\big)
= 10\,(0.01\,\theta_1^2 + 0 + \theta_2^2)
= 0.1\,\theta_1^2 + 10\,\theta_2^2
$$

Le facteur 10 est un simple choix d'échelle : il donne des coefficients ronds (0.1 et 10) sans changer le minimum ni le rapport entre les directions.

### Paramètre optimal $\theta^*$

Le paramètre optimal minimise l'**espérance** de la loss sur la distribution des données :

$$
\theta^* = \underset{\theta \in \mathbb{R}^d}{\arg\min}\; \mathbb{E}_{(x, y)}\big[\ell\big(f_\theta(x),\, y\big)\big]
$$

En pratique, cette espérance est inconnue : on la remplace par la moyenne empirique sur les $n$ images d'entraînement $(x_i, y_i)$ :

$$
\theta^* \approx \underset{\theta \in \mathbb{R}^d}{\arg\min}\; \frac{1}{n}\sum_{i=1}^{n} \ell\big(f_\theta(x_i),\, y_i\big)
$$

In [1]:
# Importation des librairies (à exécuter en premier)
import importlib

import numpy as np

import mathys_utils
importlib.reload(mathys_utils)   # recharge utils.py après une modification
from mathys_utils import (
    plot_loss_surface,
    plot_gradient_surface,
    hessian, plot_hessian_surface,
    flatness_measures, plot_flatness_box,
    plot_sensitivity,
)

<a id="etude-loss"></a>
## 2. Étude de $L = 0.1\,\theta_1^2 + 10\,\theta_2^2$

### Interprétation de la loss selon $\theta_i$

$$L(\theta_1, \theta_2) = 0.1\,\theta_1^2 + 10\,\theta_2^2$$

- **Selon $\theta_1$ (direction plate).** $\theta_1$ multiplie la moyenne $z$ de l'image, qui est petite (écart-type 0.1) et de moyenne nulle. Changer $\theta_1$ ne modifie presque pas la sortie $\hat y = \theta_1 z + \theta_2$, donc la loss monte lentement : $L = 0.9$ seulement en $\theta_1 = \pm 3$.
- **Selon $\theta_2$ (direction raide).** $\theta_2$ est un biais : il décale la sortie de la même quantité pour toutes les images. La loss monte vite : $L = 0.9$ dès $\theta_2 = \pm 0.3$, et $L = 90$ en $\theta_2 = \pm 3$.

Le minimum $\theta^* = (0, 0)$ est donc **plat dans une direction et raide dans l'autre**.

Les deux coupes ci-dessous ont les mêmes axes et la même échelle en abscisse et en ordonnée.

In [19]:
fig = plot_loss_surface()
fig.show()

<a id="gradient-hessienne"></a>
## 3. Rôles du gradient et de la Hessienne

### Comment quantifier la forme de la loss ?

Les coupes montrent la forme de la loss, mais ne la mesurent pas. Pour la quantifier, on utilise ses dérivées, qui apparaissent dans le développement de Taylor autour d'un point $\theta$ :

$$
L(\theta + \delta) \approx L(\theta) + \underbrace{\nabla L(\theta)^\top \delta}_{\text{ordre 1 : pente}} + \tfrac{1}{2}\,\underbrace{\delta^\top H\, \delta}_{\text{ordre 2 : courbure}}
$$

- la **pente** → le **gradient** (dérivée d'ordre 1) ;
- la **courbure** → la **Hessienne** (dérivée d'ordre 2).

### Le gradient : la pente dans chaque direction $\theta_i$

$$
\nabla L(\theta) = \begin{pmatrix} \dfrac{\partial L}{\partial \theta_1} \\[2mm] \dfrac{\partial L}{\partial \theta_2} \end{pmatrix}
= \begin{pmatrix} 0.2\,\theta_1 \\ 20\,\theta_2 \end{pmatrix}
$$

La composante $\dfrac{\partial L}{\partial \theta_i}$ est la **pente** de la loss dans la direction $\theta_i$ : de combien la loss monte quand on bouge un peu $\theta_i$.

### La Hessienne : la courbure du fond

$$
H = \nabla^2 L = \begin{pmatrix} \dfrac{\partial^2 L}{\partial \theta_1^2} & \dfrac{\partial^2 L}{\partial \theta_1 \partial \theta_2} \\[2mm] \dfrac{\partial^2 L}{\partial \theta_2 \partial \theta_1} & \dfrac{\partial^2 L}{\partial \theta_2^2} \end{pmatrix}
= \begin{pmatrix} 0.2 & 0 \\ 0 & 20 \end{pmatrix}
$$

Le terme $\dfrac{\partial^2 L}{\partial \theta_i^2}$ est la **courbure** de la loss dans la direction $\theta_i$ : à quelle vitesse la pente change quand on bouge $\theta_i$. Les termes croisés (nuls ici) décrivent comment les directions interagissent.

In [14]:
fig = plot_gradient_surface()
fig.show()

In [20]:
H, eigvals, eigvecs = hessian()
print("H =\n", H)                                       # [[0.2, 0], [0, 20]]
print("valeurs propres :", eigvals)                     # [0.2, 20]
print("vecteurs propres :\n", eigvecs)                  # axes theta1 et theta2
print("conditionnement :", eigvals[-1] / eigvals[0])    # 100

plot_hessian_surface().show()

H =
 [[ 0.2  0. ]
 [ 0.  20. ]]
valeurs propres : [ 0.2 20. ]
vecteurs propres :
 [[1. 0.]
 [0. 1.]]
conditionnement : 100.0


<a id="platitude"></a>
## 4. Mesurer la platitude avec $H$

Au minimum, le gradient est nul : toute l'information sur la forme du fond est dans $H$. On la résume par un seul nombre. **Plus ce nombre est petit, plus le minimum est plat.**

<a id="lambda-max"></a>
### 4.1 La plus grande valeur propre $\lambda_{\max}$

*Référence : Keskar et al., « On Large-Batch Training for Deep Learning: Generalization Gap and Sharp Minima », ICLR 2017.*

$$
\lambda_{\max}(H) = \max_{\|u\| = 1} u^\top H\, u
$$

C'est la courbure de la **direction la plus raide**. Elle donne la pire hausse de la loss pour un pas de longueur $\varepsilon$ : $\Delta L \le \tfrac{1}{2}\varepsilon^2 \lambda_{\max}$. Elle ignore toutes les autres directions.

Sur l'exemple : $\lambda_{\max} = 20$ (direction $\theta_2$).

<a id="trace"></a>
### 4.2 La trace $\mathrm{tr}(H)$

*Référence : Yao et al., « PyHessian: Neural Networks Through the Lens of the Hessian », IEEE BigData 2020.*

$$
\mathrm{tr}(H) = \sum_i \lambda_i
\qquad\text{et}\qquad
\mathbb{E}_{\delta \sim \mathcal{N}(0, \sigma^2 I)}\big[\Delta L\big] \approx \tfrac{1}{2}\,\sigma^2\, \mathrm{tr}(H)
$$

C'est la **courbure totale**, sur toutes les directions. Elle donne la hausse moyenne de la loss quand on perturbe les poids au hasard. Elle est dominée par les grandes valeurs propres.

Sur l'exemple : $\mathrm{tr}(H) = 0.2 + 20 = 20.2$.

<a id="score-b"></a>
### 4.3 Le score $\tilde B$ de *Flat Minima*

*Référence : Hochreiter & Schmidhuber, « Flat Minima », Neural Computation, 1997.*

**L'idée : la boîte.** Un minimum plat est une grande région de l'espace des poids où la loss reste presque constante. On mesure cette région par une **boîte** centrée sur $\theta^*$, alignée sur les axes, de demi-côtés $\Delta\theta_i$. C'est la plus grande boîte dans laquelle la loss monte d'au plus une tolérance $\tau$, avec la même contribution de chaque direction. On obtient :

$$
\Delta\theta_i = \sqrt{\frac{2\tau}{d\, H_{ii}}}
\qquad\qquad
\tilde B = -\sum_{i=1}^{d} \log \Delta\theta_i
$$

- $\tau$ : la hausse de loss tolérée dans la boîte (notée $\epsilon$ dans le papier) ;
- $d$ : le nombre de paramètres ;
- $H_{ii}$ : la courbure dans la direction $\theta_i$ ;
- $\Delta\theta_i$ : la demi-largeur de la boîte selon $\theta_i$. C'est la **précision** avec laquelle il faut fixer $\theta_i$ : une direction plate donne un grand $\Delta\theta_i$ ;
- $\tilde B$ : moins le log du volume de la boîte (à une constante près). C'est aussi le nombre de bits nécessaires pour décrire les poids. Un $\tilde B$ petit veut dire une grande boîte, donc un minimum plat.

Comme $\tilde B = \tfrac{1}{2}\sum_i \log H_{ii} + \text{cste}$, toutes les directions comptent, en échelle log. Une direction très plate peut compenser une direction raide.

Sur l'exemple, avec $\tau = 1.8$ : $\Delta\theta_1 = 3$, $\Delta\theta_2 = 0.3$. La boîte est $[-3, 3] \times [-0.3, 0.3]$ et chaque coin donne $L = 0.9 + 0.9 = \tau$. On obtient $\tilde B = -\log(3 \times 0.3) = -\log 0.9 \approx 0.105$.

<a id="recap"></a>
### 4.4 Récapitulatif

| Mesure | Valeur | Ce qu'elle regarde |
|---|---|---|
| $\lambda_{\max}$ | 20 | la direction la plus raide seulement |
| $\mathrm{tr}(H)$ | 20.2 | la somme des courbures |
| $\tilde B$ ($\tau = 1.8$) | 0.105 | le volume de la boîte (toutes les directions, en log) |

In [22]:
tau = 1.8
m = flatness_measures(tau=tau)
print(f"lambda_max = {m['lambda_max']:g}")             # 20
print(f"tr(H)      = {m['trace']:g}")                  # 20.2
print(f"Δθ         = {m['dtheta']}")                   # [3, 0.3]
print(f"B~         = {m['B_tilde']:.3f}")              # 0.105

plot_flatness_box(tau=tau).show()

lambda_max = 20
tr(H)      = 20.2
Δθ         = [3.  0.3]
B~         = 0.105


<a id="sensibilite"></a>
## 5. Sensibilité de la loss selon la direction

On part du minimum $\theta^* = (0, 0)$ et on fait un pas de longueur $\varepsilon$ dans une direction $u = (\cos\varphi,\ \sin\varphi)$, avec $\|u\| = 1$. Combien la loss augmente-t-elle ?

### Développement de Taylor le long de $u$

$$
L(\theta^* + \varepsilon u) = L(\theta^*) + \varepsilon\, \underbrace{\nabla L(\theta^*)^\top u}_{\text{ordre 1 : pente}} + \tfrac{1}{2}\,\varepsilon^2\, \underbrace{u^\top H\, u}_{\text{ordre 2 : courbure}}
$$

Ici la loss est quadratique : ce développement est **exact**, pas une approximation.

- **Le terme d'ordre 1 est nul** au minimum, car $\nabla L(\theta^*) = 0$. C'est ce que montrent les deux droites du gradient : elles passent par 0. La pente ne distingue donc pas les directions au minimum.
- **Le terme d'ordre 2 est le seul qui compte.** La sensibilité dans la direction $u$ est la **courbure directionnelle** :

$$
\Delta L(\varepsilon, u) = \tfrac{1}{2}\,\varepsilon^2\, k(u), \qquad k(u) = u^\top H u = 0.2\cos^2\varphi + 20\sin^2\varphi .
$$

### Les deux directions extrêmes

Les directions extrêmes sont les vecteurs propres de $H$ (ici les axes) :

| Direction | $\varphi$ | $k(u)$ | $\Delta L$ pour $\varepsilon = 0.1$ | Distance pour atteindre $\Delta L = 0.9$ |
|---|---|---|---|---|
| $\theta_1$ (plate) | 0° | $\lambda_{\min} = 0.2$ | $0.001$ | $\varepsilon = \sqrt{2 \cdot 0.9 / 0.2} = 3$ |
| $\theta_2$ (raide) | 90° | $\lambda_{\max} = 20$ | $0.1$ | $\varepsilon = \sqrt{2 \cdot 0.9 / 20} = 0.3$ |

- À pas égal, la loss monte **100 fois plus** selon $\theta_2$ que selon $\theta_1$ (rapport $\kappa = \lambda_{\max}/\lambda_{\min}$).
- À hausse de loss égale, on peut aller **10 fois plus loin** selon $\theta_1$ (rapport $\sqrt{\kappa}$). On retrouve les bornes $\pm 3$ et $\pm 0.3$ des coupes de la loss.

Pour toute autre direction, $k(u)$ est compris entre ces deux valeurs : $\lambda_{\min} \le u^\top H u \le \lambda_{\max}$ (quotient de Rayleigh).

In [25]:
fig = plot_sensitivity(eps=0.1)
fig.show()